# MonoSplat Colab GPU Launcher

Run cells **top to bottom**. Do not skip cells.

**Steps:**
1. Runtime Verification
2. Configuration
3. Repository Checkout
4. Dependency Installation
5. Dataset Upload (upload your `colab_training_package.zip` directly)
6. Discover Dataset Paths
7. Checkpoint Resume Selection
8. Pre-flight Checks
9. Launch Training
10. Output Inventory
11. Validate Outputs
12. Download Results ZIP
13. Debug Commands


---
## Cell 1 — Runtime Verification

Checks GPU availability and configures CUDA allocator settings.


In [1]:
import os
import platform
import subprocess
import sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError("No GPU detected. Choose Runtime > Change runtime type > GPU, then reconnect.")
print("\n".join(smi.stdout.splitlines()[:14]))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see CUDA in this runtime.")

props = torch.cuda.get_device_properties(0)
print(f"\nCUDA device: {props.name}")
print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
print(f"Torch: {torch.__version__}, CUDA: {torch.version.cuda}")
torch.cuda.empty_cache()
print("\n✅ Runtime OK")

# ── GPU-tier auto-preset ─────────────────────────────────────────────────────
# Selects safe training parameters based on available VRAM.
# These values are written into the environment and picked up by Cell 10
# (Launch Training) as --extra-args overrides, so config.yaml is never
# mutated on disk — the repo stays clean between runs.
#
# Tier thresholds (conservative — T4 reports 14.75 GB, not 15):
#   T4  : < 20 GB  →  safe defaults (max_gaussians=80k, no LPIPS, 18k iters)
#   L4  : 20–35 GB →  mid tier      (max_gaussians=150k, lpips=0.05, 25k iters)
#   A100: > 35 GB  →  full quality  (max_gaussians=300k, lpips=0.1,  30k iters)
#
vram_gb = props.total_memory / 1e9

if vram_gb < 20:
    GPU_TIER = "T4"
    _PRESET = dict(
        max_gaussians=80000,
        densify_until_iter=5000,
        densification_interval=500,  # FIX: was 300; matches config.yaml FIX-3 (halves realloc events on T4)
        lambda_lpips=0.0,
        iterations=18000,
        opacity_reset_interval=3000,
        position_lr_max_steps=18000,
    )
elif vram_gb < 35:
    GPU_TIER = "L4"
    _PRESET = dict(
        max_gaussians=150000,
        densify_until_iter=8000,
        densification_interval=200,
        lambda_lpips=0.05,
        iterations=25000,
        opacity_reset_interval=4000,
        position_lr_max_steps=25000,
    )
else:
    GPU_TIER = "A100"
    _PRESET = dict(
        max_gaussians=300000,
        densify_until_iter=15000,
        densification_interval=100,
        lambda_lpips=0.1,
        iterations=30000,
        opacity_reset_interval=5000,
        position_lr_max_steps=30000,
    )

# Serialise as CLI override string consumed by Cell 10
import shlex
_OVERRIDE_ARGS = " ".join(
    f"--training.{k} {v}" for k, v in _PRESET.items()
)
os.environ["MONOSPLAT_EXTRA_TRAIN_ARGS"] = _OVERRIDE_ARGS

print(f"\n{'='*55}")
print(f"  GPU tier : {GPU_TIER}  ({vram_gb:.1f} GB VRAM)")
print(f"  Preset applied:")
for k, v in _PRESET.items():
    print(f"    {k:<30} = {v}")
print(f"{'='*55}")
print("\n✅ GPU preset applied — Cell 10 will use these values.")


Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Tue Jun  2 00:58:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |         

In [2]:
# ── GPU Diagnostic ────────────────────────────────────────────────────────
# Run this before training to confirm the GPU is healthy and VRAM is free.
import torch

if torch.cuda.is_available():
    device_name  = torch.cuda.get_device_name(0)
    vram_total   = torch.cuda.get_device_properties(0).total_memory / 1024**3
    vram_alloc   = torch.cuda.memory_allocated(0) / 1024**3
    vram_reserv  = torch.cuda.memory_reserved(0)  / 1024**3
    print(f"GPU           : {device_name}")
    print(f"VRAM total    : {vram_total:.1f} GB")
    print(f"VRAM allocated: {vram_alloc:.2f} GB")
    print(f"VRAM reserved : {vram_reserv:.2f} GB")
else:
    print("WARNING: No CUDA GPU found. Set Runtime → Change runtime type → T4 GPU.")


GPU           : Tesla T4
VRAM total    : 14.6 GB
VRAM allocated: 0.00 GB
VRAM reserved : 0.00 GB


In [3]:
# ── VRAM Cleanup (run before training to maximise free memory) ────────────
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free  = torch.cuda.mem_get_info(0)[0] / 1024**3
    total = torch.cuda.mem_get_info(0)[1] / 1024**3
    print(f"VRAM free : {free:.1f} GB / {total:.1f} GB")
    print("Cache cleared — ready to train.")
else:
    print("No CUDA device — skipping cache flush.")


VRAM free : 14.5 GB / 14.6 GB
Cache cleared — ready to train.


---
## Cell 2 — Configuration

Defines all path variables used by every subsequent cell. Edit `GITHUB_REPO` if your repo URL differs.


In [4]:
from pathlib import Path

# ── Repository ─────────────────────────────────────────────────────────
GITHUB_REPO      = "https://github.com/Somaskandan931/Monosplat.git"  # ← edit if needed
REPO_DIR         = Path("/content/monosplat")
TRAIN_ENTRYPOINT = REPO_DIR / "colab" / "train.py"    # train.py lives in colab/, not scripts/
CONFIG_PATH      = REPO_DIR / "configs" / "config.yaml"
EXPORT_SCRIPT    = REPO_DIR / "colab" / "export_splat.py"  # export_splat.py also lives in colab/

# ── Runtime working directory ───────────────────────────────────────────
WORK_ROOT = Path("/content/monosplat_runtime")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"REPO_DIR        : {REPO_DIR}")
print(f"TRAIN_ENTRYPOINT: {TRAIN_ENTRYPOINT}")
print(f"CONFIG_PATH     : {CONFIG_PATH}")
print(f"WORK_ROOT       : {WORK_ROOT}")
print("\n✅ Configuration set")


REPO_DIR        : /content/monosplat
TRAIN_ENTRYPOINT: /content/monosplat/scripts/train.py
CONFIG_PATH     : /content/monosplat/configs/config.yaml
WORK_ROOT       : /content/monosplat_runtime

✅ Configuration set


---
## Cell 3 — Repository Checkout

Clones or pulls the latest repository code.


In [5]:
import subprocess

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Repository already cloned — pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {GITHUB_REPO} -> {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)

result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True
)
print(f"Commit: {result.stdout.strip()}")
print("\n✅ Repository ready")


Repository already cloned — pulling latest changes...
Commit: 1a6af34

✅ Repository ready


---
## Cell 4 — Dependency Installation

Installs packages from the repository requirements file.


In [6]:
import importlib
import os
import subprocess
import sys

os.chdir(REPO_DIR)

# ── 1. Install gsplat FIRST, matched to Colab's CUDA version ────────────────
# Must happen before requirements.txt so the correct wheel is already cached.
try:
    import gsplat  # skip if already installed in this session
    print(f"  ✓ gsplat already present ({getattr(gsplat, '__version__', 'unknown')})")
except ImportError:
    try:
        import torch
        cuda_ver = torch.version.cuda or ""          # e.g. "12.1"
        cuda_tag = "cu" + cuda_ver.replace(".", "")  # → "cu121"
        # FIX: [:4] slices a trailing dot on versions like "2.6.0+cu121" → "2.6." → bad tag "pt26."
        # Safe approach: strip build suffix first, then take major+minor digits only.
        torch_tag = "pt" + "".join(torch.__version__.split("+")[0].split(".")[:2])  # → "pt26", "pt211"
        index_url = f"https://docs.gsplat.studio/whl/{torch_tag}{cuda_tag}"
        print(f"  Installing gsplat for {torch_tag}/{cuda_tag} …")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "gsplat", "--extra-index-url", index_url],
            check=True,
        )
    except Exception as exc:
        print(f"  ⚠ gsplat wheel install failed: {exc} (fallback renderer will be used)")

# ── 2. Install the rest of the requirements ──────────────────────────────────
requirements = REPO_DIR / "requirements-colab.txt"
if not requirements.exists():
    requirements = REPO_DIR / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(f"No requirements file found under {REPO_DIR}")

print(f"Installing from: {requirements.name}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
    check=True,
)

# ── 2b. Install lpips (perceptual loss — not in requirements.txt) ───────────
try:
    import lpips  # skip if already installed
    print("  ✓ lpips already present")
except ImportError:
    print("  Installing lpips...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lpips"],
        check=True,
    )
    print("  ✓ lpips installed")

# ── 3. Verify core imports ───────────────────────────────────────────────────
for module_name in ["torch", "yaml", "PIL", "numpy", "tqdm"]:
    importlib.import_module(module_name)
    print(f"  ✓ {module_name}")

try:
    import gsplat
    print(f"  ✓ gsplat {getattr(gsplat, '__version__', 'unknown')}")
except Exception as exc:
    print(f"  ⚠ gsplat not importable: {exc} (repository will use fallback renderer)")

print("\n✅ Dependencies installed")

  ✓ gsplat already present (1.5.3)
Installing from: requirements-colab.txt
  ✓ lpips already present
  ✓ torch
  ✓ yaml
  ✓ PIL
  ✓ numpy
  ✓ tqdm
  ✓ gsplat 1.5.3

✅ Dependencies installed


---
## Cell 5 — Google Drive Mount (optional)

Mount Drive **only** if you want checkpoints / logs saved for long-term storage.
Skipping this step is fine — training and export work entirely in `/content`.


In [7]:
from pathlib import Path
import time

DRIVE_MOUNTED = False
DRIVE_ROOT = None

try:
    from google.colab import drive
    print("Mounting Google Drive — follow the authentication prompt...")
    drive.mount("/content/drive", force_remount=True)
    DRIVE_ROOT = Path("/content/drive/MyDrive/MonoSplat")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_MOUNTED = True
    print("✅ Drive mounted at", DRIVE_ROOT)
except Exception as exc:
    print(f"ℹ  Drive not mounted ({exc}) — outputs will stay in /content and downloaded directly.")
    print("   This is fine. Proceed to the next cell.")


Mounting Google Drive — follow the authentication prompt...
Mounted at /content/drive
✅ Drive mounted.
Drive root: /content/drive/MyDrive/MonoSplat
  exports/      ← final.ply, final.splat
  checkpoints/  ← training checkpoints
  logs/         ← training log
  previews/     ← rendered preview images

✅ Drive ready


---
## Cell 6 — Dataset Upload

Upload the `colab_training_package.zip` you downloaded from the MonoSplat desktop app.

Run the cell below — a file-picker will appear. Select your ZIP and wait for the upload to finish.


In [8]:
import os
import zipfile
from pathlib import Path
from google.colab import files

print("Select your colab_training_package.zip in the file picker below...")
uploaded = files.upload()

zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if not zip_names:
    raise FileNotFoundError("No .zip file was uploaded. Re-run this cell and select your ZIP.")

archive_path = Path("/content") / zip_names[0]
if not archive_path.exists():
    # google.colab uploads to cwd
    archive_path = Path.cwd() / zip_names[0]

if not zipfile.is_zipfile(archive_path):
    raise zipfile.BadZipFile(f"{archive_path.name} is not a valid zip archive.")

# ── Extract ──────────────────────────────────────────────────────────────────
WORK_ROOT = Path("/content/monosplat_data")
job_stem  = archive_path.stem.replace("_for_colab", "").replace("colab_training_package", "job")
JOB_ROOT  = WORK_ROOT / job_stem
JOB_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Extracting {archive_path.name} → {JOB_ROOT} ...")
with zipfile.ZipFile(archive_path) as zf:
    for member in zf.infolist():
        member.filename = member.filename.replace("\\", "/")
        zf.extract(member, JOB_ROOT)

# Unwrap single nested directory (e.g. zip contained job_stem/frames/)
nested = JOB_ROOT / job_stem
if nested.exists() and nested.is_dir():
    JOB_ROOT = nested

print(f"JOB_ROOT: {JOB_ROOT}")
print("Contents:", [p.name for p in JOB_ROOT.iterdir()])
print("\n✅ Dataset staged")


Extracting colab_training_package.zip -> /content/monosplat_runtime/colab_training_package
JOB_ROOT: /content/monosplat_runtime/colab_training_package

✅ Dataset staged


---
## Cell 7 — Discover Dataset Paths

Finds the COLMAP sparse model and frames directory inside the extracted job.


In [9]:
import os
import sys
from pathlib import Path

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from utils.config_loader import load_config
cfg = load_config(str(CONFIG_PATH))

def _normalise_images_txt(colmap_dir):
    """If the COLMAP dir has frames.txt but not images.txt, create a symlink."""
    images = colmap_dir / "images.txt"
    frames = colmap_dir / "frames.txt"
    if not images.exists() and frames.exists():
        images.symlink_to(frames)
        print(f"  ℹ  Aliased frames.txt -> images.txt in {colmap_dir}")

def find_colmap_dir(root):
    """Search for a COLMAP text model under root.

    Handles all common export layouts:
      sparse_text/          (MonoSplat default)
      sparse/0/             (standard COLMAP)
      sparse/               (flat export)
      colmap/sparse_text/   (nested colmap/ prefix)
      colmap/sparse/0/      (nested colmap/ prefix)
    Also accepts frames.txt as an alias for images.txt.
    """
    patterns = [
        "**/sparse_text",
        "**/sparse/0",
        "**/sparse",
        "**/colmap/sparse_text",
        "**/colmap/sparse/0",
        "**/colmap/sparse",
    ]
    for pattern in patterns:
        for path in sorted(Path(root).glob(pattern)):
            has_images = (path / "images.txt").exists() or (path / "frames.txt").exists()
            if has_images and (path / "cameras.txt").exists() and (path / "points3D.txt").exists():
                _normalise_images_txt(path)
                return path
    return None

def find_image_dir(root):
    for name in ["frames", "processed", "images", "input"]:
        for path in sorted(Path(root).glob(f"**/{name}")):
            if path.is_dir() and any(
                f.suffix.lower() in {".png", ".jpg", ".jpeg"} for f in path.iterdir()
            ):
                return path
    # Fallback: any directory containing images
    for path in sorted(Path(root).glob("**/*")):
        if path.is_dir() and path.name not in {"__pycache__", ".git", "sparse_text", "sparse"}:
            try:
                if any(f.suffix.lower() in {".png", ".jpg", ".jpeg"} for f in path.iterdir()):
                    return path
            except PermissionError:
                pass
    return None

SOURCE_PATH = find_colmap_dir(JOB_ROOT)
IMAGE_DIR   = find_image_dir(JOB_ROOT)

if SOURCE_PATH is None:
    # Print the tree so the user can see what was actually extracted
    print(f"Contents of JOB_ROOT ({JOB_ROOT}):")
    for p in sorted(JOB_ROOT.rglob("*"))[:60]:
        indent = "  " * len(p.relative_to(JOB_ROOT).parts)
        print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")
    raise FileNotFoundError(
        f"COLMAP text model not found under {JOB_ROOT}\n"
        "Expected cameras.txt + images.txt (or frames.txt) + points3D.txt in one of:\n"
        "  sparse_text/  |  sparse/0/  |  sparse/  |  colmap/sparse_text/  |  colmap/sparse/0/"
    )

if IMAGE_DIR is None:
    raise FileNotFoundError(
        f"Training images not found under {JOB_ROOT}\n"
        "Expected a frames/ or processed/ subdirectory with .jpg/.png files."
    )

JOB_ID     = os.environ.get("MONOSPLAT_JOB_ID") or Path(JOB_ROOT).name
OUTPUT_DIR = WORK_ROOT / JOB_ID / "models" / "gaussian"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

img_count = sum(1 for f in IMAGE_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})

print(f"Source path (COLMAP) : {SOURCE_PATH}")
print(f"Image dir            : {IMAGE_DIR}")
print(f"Output dir           : {OUTPUT_DIR}")
print(f"Job ID               : {JOB_ID}")
print(f"sh_degree (config)   : {cfg.get('model', {}).get('sh_degree', 'N/A')}")
print(f"Training images      : {img_count}")

if img_count < 10:
    raise RuntimeError(f"Too few training images: {img_count}. MonoSplat requires at least 10.")
elif img_count > 250:
    print(f"\n\u26a0  Large dataset: {img_count} images detected.")
    print("   Recommended for Colab T4: 150–250 frames, 720p, frame stride 2–4.")
    print("   Training iterations will be auto-capped at 18,000 by colab/train.py.")
else:
    print(f"\u2713 Dataset size OK ({img_count} images)")

print("\n✅ Dataset paths verified")


Source path (COLMAP) : /content/monosplat_runtime/colab_training_package/sparse_text
Image dir            : /content/monosplat_runtime/colab_training_package/frames
Output dir           : /content/monosplat_runtime/colab_training_package/models/gaussian
Job ID               : colab_training_package
sh_degree (config)   : 3
Training images      : 75
✓ Dataset size OK (75 images)

✅ Dataset paths verified


---
## Cell 8 — Checkpoint Resume Selection

Auto-selects the latest checkpoint if one exists, enabling resume from a previous run.


In [10]:
import os
import re

# Set FORCE_FRESH=True to always start from scratch (recommended).
# Only set False to resume a *partial* run — resuming a completed checkpoint
# (iter == total iterations) produces zero training iterations.
FORCE_FRESH = True  # ← change to False only to resume a partial checkpoint

RESUME_CHECKPOINT = None

if not FORCE_FRESH:
    RESUME_CHECKPOINT = os.environ.get("MONOSPLAT_RESUME")

    if not RESUME_CHECKPOINT:
        checkpoint_roots = [
            OUTPUT_DIR / "checkpoints",
            DRIVE_ROOT / "checkpoints" / JOB_ID,
        ]
        checkpoint_files = sorted(
            {p for root in checkpoint_roots if root.exists()
               for ext in ["*.pkl", "*.pt", "*.ckpt"]
               for p in root.glob(ext)},
            key=lambda p: p.stat().st_mtime
        )
        if checkpoint_files:
            RESUME_CHECKPOINT = str(checkpoint_files[-1])

    if RESUME_CHECKPOINT:
        match = re.search(r'checkpoint_(\d+)', str(RESUME_CHECKPOINT))
        if match:
            ckpt_iter   = int(match.group(1))
            train_iters = cfg.get('training', {}).get('iterations', 20000)
            if ckpt_iter >= train_iters:
                print(
                    f"⚠  Checkpoint at iter {ckpt_iter:,} == training limit {train_iters:,}.\n"
                    f"   Resuming would produce ZERO training iterations.\n"
                    f"   Setting FORCE_FRESH=True automatically."
                )
                RESUME_CHECKPOINT = None
            else:
                print(f"Resuming from iter {ckpt_iter:,} / {train_iters:,}: {RESUME_CHECKPOINT}")

if RESUME_CHECKPOINT is None:
    print("Starting FRESH — no checkpoint will be loaded.")

print("\n✅ Checkpoint selection done")


Starting FRESH — no checkpoint will be loaded.

✅ Checkpoint selection done


---
## Cell 9 — Pre-flight Checks

Validates training inputs and verifies hyperparameters before launch.
Catches the most common causes of SIGFPE crashes and zero-iteration runs.


In [11]:
import os
import sys
from pathlib import Path

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from utils.config_loader import load_config
cfg = load_config(str(CONFIG_PATH))

# ── 1. Verify COLMAP / image file alignment ─────────────────────────────
images_txt = SOURCE_PATH / 'images.txt'
if not images_txt.exists() and (SOURCE_PATH / 'frames.txt').exists():
    images_txt = SOURCE_PATH / 'frames.txt'
colmap_names = set()
if images_txt.exists():
    with open(images_txt) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) == 10:
                try:
                    int(parts[0]); float(parts[1])
                    colmap_names.add(parts[9])
                except ValueError:
                    pass

disk_names  = {p.name for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}}
missing     = colmap_names - disk_names
missing_pct = len(missing) / max(len(colmap_names), 1) * 100

print(f"COLMAP registered images : {len(colmap_names)}")
print(f"Images on disk           : {len(disk_names)}")
print(f"Missing from disk        : {len(missing)} ({missing_pct:.0f}%)")

if missing_pct > 50:
    raise RuntimeError(
        f"HARD FAILURE: {missing_pct:.0f}% of COLMAP images are missing from {IMAGE_DIR}.\n"
        "The COLMAP sparse model and the frames directory are from different runs.\n"
        "Re-extract your zip and ensure both sparse_text/ and frames/ are present."
    )
elif missing_pct > 20:
    print(f"⚠  WARNING: {missing_pct:.0f}% missing — training will use only matched images.")
else:
    print("✓ Image alignment OK")

# ── 2. Check initial Gaussian count is non-trivial ───────────────────────
pts3d = SOURCE_PATH / 'points3D.txt'
n_pts = sum(1 for l in open(pts3d) if l.strip() and not l.startswith('#')) if pts3d.exists() else 0
print(f"\nInitial 3D points (sparse cloud): {n_pts:,}")
if n_pts < 500:
    raise RuntimeError(
        f"Only {n_pts} 3D points — COLMAP reconstruction is too sparse for training.\n"
        "Re-run the preprocessing pipeline with better input footage."
    )
elif n_pts < 2000:
    print("⚠  Sparse cloud is thin (<2k pts) — expect low Gaussian count and foggy output.")
else:
    print("✓ Sparse cloud density OK")

# ── 3. Verify key config values ─────────────────────────────────────────
# FIX: cfg is a plain dict — use dict access, not dot notation
t = cfg.get("training", {})
r = cfg.get("renderer", cfg.get("model", {}))  # fall back to "model" if "renderer" key absent

print("\nActive training config:")
print(f"  opacity_reset_interval : {t.get('opacity_reset_interval', 'N/A')}  (safe value: 3000)")
print(f"  lambda_opacity_reg     : {t.get('lambda_opacity_reg', 'N/A')}  (safe value: 0.005)")
print(f"  densify_grad_threshold : {t.get('densify_grad_threshold', 'N/A')}")
print(f"  percent_dense          : {t.get('percent_dense', 'N/A')}")
print(f"  iterations             : {t.get('iterations', 'N/A'):,}  (Colab T4 cap: 18,000)")
print(f"  max_gaussians          : {t.get('max_gaussians', r.get('max_gaussians', 'N/A'))}")

if t.get('opacity_reset_interval', 9999) < 3000:
    print("\n⚠  WARNING: opacity_reset_interval < 3000.")
    print("   Edit configs/config.yaml: opacity_reset_interval: 3000")
    print("   Values below 3000 caused mass-prune crashes at iter 1600.")

if t.get('lambda_opacity_reg', 0) > 0.005:
    print("\n⚠  WARNING: lambda_opacity_reg > 0.005.")
    print("   Edit configs/config.yaml: lambda_opacity_reg: 0.002")
    print("   High values suppress all opacities to zero before the prune pass.")

if missing_pct > 20:
    print(
        f"\n⚠  DENSITY WARNING: {missing_pct:.0f}% of COLMAP images are missing from disk.\n"
        f"   Only {len(colmap_names) - len(missing)}/{len(colmap_names)} cameras will train.\n"
        "   If clone=0 persists past iter 2000, re-run preprocessing with the full frame set."
    )

# ── 4. Verify normalize_scene module ───────────────────────────────────────
normalize_scene_path = REPO_DIR / "src" / "preprocessing" / "normalize_scene.py"
if normalize_scene_path.exists():
    print("✓ normalize_scene.py found — scene normalization enabled")
else:
    print("⚠  normalize_scene.py missing from src/preprocessing/ — pull latest repo code")

print("\n✅ Pre-flight checks passed — ready to train")

COLMAP registered images : 75
Images on disk           : 75
Missing from disk        : 0 (0%)
✓ Image alignment OK

Initial 3D points (sparse cloud): 83,719
✓ Sparse cloud density OK

Active training config:
  opacity_reset_interval : 1000  (safe value: 3000)
  lambda_opacity_reg     : 0.001  (safe value: 0.005)
  densify_grad_threshold : 0.0003
  percent_dense          : 0.01
  iterations             : 30,000  (Colab T4 cap: 18,000)
  max_gaussians          : 150000

⚠  WARNING: opacity_reset_interval < 3000.
   Edit configs/config.yaml: opacity_reset_interval: 3000
   Values below 3000 caused mass-prune crashes at iter 1600.
✓ normalize_scene.py found — normalization enabled

✅ Pre-flight checks passed — ready to train


---
## Cell 10 — Launch Training

Runs the training script. Output streams live to the console and is saved to Drive.


In [ ]:
# Cell 10 — Launch Training

import os
import shlex
import subprocess
import sys

os.chdir(REPO_DIR)

# ── Clean stale outputs (avoids reusing bad checkpoints) ────────────────
# FORCE_FRESH is set in Cell 8.  If the user chose to resume, skip the wipe.
if FORCE_FRESH:
    import shutil as _shutil
    for _stale in [OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "logs"]:
        if _stale.exists():
            _shutil.rmtree(str(_stale))
            print(f"  🗑  Cleared {_stale.relative_to(OUTPUT_DIR)}")
    print("  ✓ Stale outputs cleared — starting fresh.")
else:
    print("  ℹ  Resume mode — keeping existing checkpoints.")

# ── Show active GPU preset so the log captures it ───────────────────────
_active_preset = os.environ.get("MONOSPLAT_EXTRA_TRAIN_ARGS", "")
if _active_preset:
    print("Active GPU preset overrides:")
    for _arg in _active_preset.split("--")[1:]:
        print(f"  --{_arg.strip()}")
    print()
else:
    print("No GPU preset overrides — using config.yaml defaults.")
    print()

cmd = [
    sys.executable,
    str(TRAIN_ENTRYPOINT),
    "--sparse", str(SOURCE_PATH),
    "--frames", str(IMAGE_DIR),
    "--output", str(OUTPUT_DIR),
    "--config", str(CONFIG_PATH),
]

if RESUME_CHECKPOINT:
    cmd.extend(["--resume", str(RESUME_CHECKPOINT)])

# NOTE: MONOSPLAT_EXTRA_TRAIN_ARGS is already read by train.py internally
# via _apply_env_overrides(cfg) — do NOT also pass it on the CLI or argparse
# will reject the dotted --training.key format and exit with code 2.
# The env var alone is sufficient; train.py logs every override it applies.

print("Launching:")
print("  " + " ".join(shlex.quote(p) for p in cmd))
print(f"  resume : {RESUME_CHECKPOINT or 'NONE (fresh start)'}")
print()

env = os.environ.copy()
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    f"{REPO_DIR / 'src'}:{REPO_DIR}"
    if not existing
    else f"{REPO_DIR / 'src'}:{REPO_DIR}:{existing}"
)
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
env["CUDA_LAUNCH_BLOCKING"] = "0"
env["TORCH_CUDNN_V8_API_ENABLED"] = "1"

# FIX: DRIVE_ROOT is None when Drive is not mounted — fall back to local log
if DRIVE_MOUNTED and DRIVE_ROOT is not None:
    TRAIN_LOG = DRIVE_ROOT / "logs" / f"{JOB_ID}_train.log"
else:
    TRAIN_LOG = OUTPUT_DIR / "logs" / f"{JOB_ID}_train.log"
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

with open(TRAIN_LOG, "w", encoding="utf-8") as log_file:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in proc.stdout:
        print(line, end="", flush=True)
        log_file.write(line)

    return_code = proc.wait()
    log_file.flush()
    os.fsync(log_file.fileno())

_EXIT_HINTS = {
    -8: (
        "SIGFPE - floating-point exception in CUDA kernel.\n"
        "Cause: CUDA rasterization failed, often after unstable densification/pruning.\n"
        "Fix: pull the latest repo code, then rerun Cell 10. If it repeats, lower max_gaussians in configs/config.yaml."
    ),
    -9: (
        "SIGKILL - process killed by OS, usually OOM.\n"
        "Fix: reduce max_gaussians in configs/config.yaml and rerun."
    ),
    -11: (
        "SIGSEGV - segmentation fault in native code.\n"
        "Usually a gsplat version mismatch. Rerun dependency install."
    ),
    1: "Python exception - check the traceback above.",
    2: "Argument error - a flag or path was rejected by train.py.",
}

if return_code != 0:
    hint = _EXIT_HINTS.get(return_code, f"Unexpected exit code {return_code}.")
    log_text = TRAIN_LOG.read_text(encoding="utf-8", errors="replace")
    last_lines = "\n".join(log_text.splitlines()[-30:])
    raise RuntimeError(
        f"Training failed (exit {return_code}):\n"
        f"{hint}\n\n"
        f"--- Last 30 log lines ---\n{last_lines}\n"
        f"Full log: {TRAIN_LOG}"
    )

print(f"\n✅ Training complete. Log: {TRAIN_LOG}")

  ✓ Stale outputs cleared — starting fresh.
Active GPU preset overrides:
  --training.max_gaussians 80000
  --training.densify_until_iter 5000
  --training.densification_interval 500
  --training.lambda_lpips 0.0
  --training.iterations 18000
  --training.opacity_reset_interval 3000
  --training.position_lr_max_steps 18000

Launching:
  /usr/bin/python3 /content/monosplat/scripts/train.py --sparse /content/monosplat_runtime/colab_training_package/sparse_text --frames /content/monosplat_runtime/colab_training_package/frames --output /content/monosplat_runtime/colab_training_package/models/gaussian --config /content/monosplat/configs/config.yaml
  resume : NONE (fresh start)

2026-06-02 01:00:56,223 [INFO] __main__ — [train] Applying env overrides: --training.max_gaussians 80000 --training.densify_until_iter 5000 --training.densification_interval 500 --training.lambda_lpips 0.0 --training.iterations 18000 --training.opacity_reset_interval 3000 --training.position_lr_max_steps 18000
2026-

---
## Cell 10b — Render Results

Renders novel-view images from the trained Gaussian model.
**Always pass `--config`** so the renderer uses the same SH degree and settings as training.
Run this after training completes.

In [ ]:
# Cell 10b — Render Results
# Renders from the trained model with the same config used during training.
# --config ensures sh_degree, bg_color, and other renderer settings match.

import os
import shlex
import subprocess
import sys

os.chdir(REPO_DIR)

RENDER_SCRIPT = REPO_DIR / "scripts" / "render.py"

if not RENDER_SCRIPT.exists():
    print("⚠  scripts/render.py not found — skipping render step.")
    print("   Novel-view previews are generated in the Output Inventory cell instead.")
else:
    render_cmd = [
        sys.executable,
        str(RENDER_SCRIPT),
        "--model_path", str(OUTPUT_DIR),
        "--config",     str(CONFIG_PATH),   # IMPORTANT: keeps SH degree + settings in sync
    ]

    print("Rendering with:")
    print("  " + " ".join(shlex.quote(p) for p in render_cmd))
    print()

    env = os.environ.copy()
    existing = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = (
        f"{REPO_DIR / 'src'}:{REPO_DIR}"
        if not existing
        else f"{REPO_DIR / 'src'}:{REPO_DIR}:{existing}"
    )

    result = subprocess.run(
        render_cmd,
        cwd=str(REPO_DIR),
        env=env,
    )

    if result.returncode == 0:
        print("\n✅ Render complete — check OUTPUT_DIR/renders/ for images.")
    else:
        print(f"\n⚠  Render exited with code {result.returncode}.")
        print("   Non-fatal — use the Output Inventory cell or SuperSplat for viewing.")


---
## Cell 11 — Output Inventory

Lists produced artifacts and shows the latest preview image.


In [ ]:
import os
import subprocess
import sys
import random
from pathlib import Path
from IPython.display import Image as IPImage, display

# ── 1. Ensure exports exist ──────────────────────────────────────────────
exports_dir = OUTPUT_DIR / "exports"
final_ply   = exports_dir / "final.ply"
final_splat = exports_dir / "final.splat"

if not (final_ply.exists() and final_splat.exists()):
    ckpts = sorted((OUTPUT_DIR / "checkpoints").glob("*.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in {OUTPUT_DIR / 'checkpoints'}")
    latest_ckpt = ckpts[-1]
    print(f"Exporting final artifacts from {latest_ckpt.name}...")
    subprocess.run(
        [sys.executable, str(EXPORT_SCRIPT),
         "--checkpoint", str(latest_ckpt),
         "--output", str(exports_dir)],
        check=True, cwd=str(REPO_DIR),
    )

# ── 2. Output inventory ──────────────────────────────────────────────────
files_found = sorted(f for f in OUTPUT_DIR.rglob("*") if f.is_file())
print(f"Output directory: {OUTPUT_DIR}\n")
for path in files_found:
    size_mb = path.stat().st_size / 1e6
    print(f"  {path.relative_to(OUTPUT_DIR)}  ({size_mb:.2f} MB)")

# ── 3. Render a novel-view preview from the trained model ────────────────
print("\nRendering preview from trained Gaussians...")
previews_dir = OUTPUT_DIR / "previews"
previews_dir.mkdir(parents=True, exist_ok=True)

try:
    import torch
    import numpy as np
    from PIL import Image

    os.chdir(REPO_DIR)
    for p in [str(REPO_DIR), str(REPO_DIR / 'src')]:
        if p not in sys.path:
            sys.path.insert(0, p)

    from reconstruction.gaussian_model import GaussianModel
    from renderer.renderer import GaussianRenderer
    from renderer.camera import Camera
    from utils.config_loader import load_config
    # FIX 1: correct function names — aliases added to colmap_utils.py
    from utils.colmap_utils import read_cameras, read_images

    cfg = load_config(str(CONFIG_PATH))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load latest checkpoint
    ckpts = sorted((OUTPUT_DIR / 'checkpoints').glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError('No checkpoints found')
    latest_ckpt = ckpts[-1]
    print(f"  Loading checkpoint: {latest_ckpt.name}")

    # FIX: GaussianModel.__init__(self, sh_degree: int = 3) — takes an int, NOT a cfg dict.
    # Passing the dict makes sh_degree a dict object → nonsense SH or crash in nn.Module init.
    model = GaussianModel(sh_degree=cfg["model"]["sh_degree"])
    ckpt_data = torch.load(str(latest_ckpt), map_location=device)
    # FIX 6: correct checkpoint key is "model" (set by trainer._save_checkpoint)
    state = ckpt_data.get('model', ckpt_data.get('model_state_dict', ckpt_data.get('gaussians', ckpt_data)))
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    model.eval()
    # FIX 5: model.num_gaussians property now exists (added to GaussianModel)
    print(f"  Loaded {model.num_gaussians:,} Gaussians")

    # Load a reference training camera to use its intrinsics
    # FIX 1: use read_cameras / read_images (correct names)
    colmap_cameras = read_cameras(str(SOURCE_PATH / 'cameras.txt'))
    images_txt = (SOURCE_PATH / 'images.txt') if (SOURCE_PATH / 'images.txt').exists() \
                 else (SOURCE_PATH / 'frames.txt')
    colmap_images  = read_images(str(images_txt))

    cam_entries = list(colmap_cameras.values())
    img_entries = list(colmap_images.values())
    ref_cam = cam_entries[0]

    # Pick a training view near the middle of the sequence for the preview
    mid_img = img_entries[len(img_entries) // 2]
    # FIX 2: Camera.from_colmap(img, cam, width, height) — no 'device' arg
    camera = Camera.from_colmap(
        mid_img, ref_cam,
        width=ref_cam.width, height=ref_cam.height,
    )

    # FIX 3: GaussianRenderer(width, height, bg_color, device) — not cfg dict
    renderer = GaussianRenderer(
        width=ref_cam.width,
        height=ref_cam.height,
        bg_color=(1.0, 1.0, 1.0),   # white background for preview
        device=str(device),
    )

    with torch.no_grad():
        # FIX 4: renderer.render() returns np.ndarray (H,W,3) uint8 directly
        img_np = renderer.render(model, camera)

    pil_img = Image.fromarray(img_np)
    preview_path = previews_dir / f"preview_{latest_ckpt.stem}.png"
    pil_img.save(str(preview_path))
    print(f"  Saved preview: {preview_path}")

    # Also save to Drive previews/ folder
    if DRIVE_MOUNTED:
        import shutil
        PREVIEW_DEST = DRIVE_ROOT / 'previews'
        PREVIEW_DEST.mkdir(parents=True, exist_ok=True)
        drive_preview = PREVIEW_DEST / f"{JOB_ID}_{preview_path.name}"
        shutil.copy2(str(preview_path), str(drive_preview))
        print(f"  ✓ Preview saved to Drive: {drive_preview.name}")

    display(IPImage(filename=str(preview_path)))
    print("\n✅ Preview rendered and displayed.")

except Exception as e:
    import traceback
    print(f"  ⚠ Preview render failed: {e}")
    traceback.print_exc()
    print("    This is non-fatal. Use final.splat in SuperSplat to view your scene.")


---
## Cell 12 — Validate Outputs

Runs the repository validation script to confirm all expected artifacts exist.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "validate_outputs.py"),
     "--model_path", str(OUTPUT_DIR)],
    capture_output=True, text=True, cwd=str(REPO_DIR)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    print("⚠  Some output validation checks failed — see above.")
else:
    print("✅ Output validation passed.")


---
## Cell 13 — Save Outputs

**Option A — Google Drive** (if mounted in Cell 5): copies exports, checkpoints, and logs to Drive.

**Option B — Direct download**: the next cell packages `exports/` into a ZIP and downloads it to your computer.

Run whichever option suits you. Both can be run.


In [ ]:
import shutil
import os
from pathlib import Path

if not DRIVE_MOUNTED:
    print("ℹ  Drive not mounted — skipping Drive sync. Use the download cell below instead.")
else:
    def _copy_with_flush(src: Path, dst: Path):
        shutil.copy2(str(src), str(dst))
        with open(dst, 'rb') as fh:
            os.fsync(fh.fileno())

    EXPORTS   = DRIVE_ROOT / "exports"
    CKPTS_DST = DRIVE_ROOT / "checkpoints" / JOB_ID
    LOGS_DST  = DRIVE_ROOT / "logs" / JOB_ID
    for d in (EXPORTS, CKPTS_DST, LOGS_DST):
        d.mkdir(parents=True, exist_ok=True)

    exports_dir = OUTPUT_DIR / "exports"
    copied = []
    for suffix in [".ply", ".splat"]:
        for f in exports_dir.rglob(f"*{suffix}"):
            dst = EXPORTS / f.name
            _copy_with_flush(f, dst)
            copied.append(dst)
            print(f"  ✓ {f.name} → {dst}")

    for ckpt in sorted((OUTPUT_DIR / "checkpoints").glob("*.ckpt")):
        _copy_with_flush(ckpt, CKPTS_DST / ckpt.name)
        print(f"  ✓ {ckpt.name} → Drive checkpoints")

    train_log = OUTPUT_DIR / "logs" / "train_log.txt"
    if train_log.exists():
        _copy_with_flush(train_log, LOGS_DST / "train_log.txt")
        print(f"  ✓ train_log.txt → Drive logs")

    print("\n✅ Outputs synced to Google Drive")


---
## Cell 13b — Download Results ZIP

Packages `exports/final.ply` and `exports/final.splat` into a single ZIP
and triggers a browser download. Use this if you skipped Google Drive.


In [ ]:
import shutil
import zipfile
from pathlib import Path
from google.colab import files

exports_dir  = OUTPUT_DIR / "exports"
results_zip  = Path("/content/monosplat_results.zip")

if not exports_dir.exists():
    raise FileNotFoundError(
        f"exports/ directory not found at {exports_dir}. "
        "Run Cell 11 (Output Inventory) first to generate the exports."
    )

artifacts = list(exports_dir.rglob("*.ply")) + list(exports_dir.rglob("*.splat"))
if not artifacts:
    raise FileNotFoundError("No .ply or .splat files found in exports/. Training may not have completed.")

print("Packaging exports:")
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in artifacts:
        arc_name = f"exports/{f.name}"
        zf.write(f, arc_name)
        print(f"  + {arc_name}  ({f.stat().st_size / 1e6:.1f} MB)")

print(f"\nZIP size: {results_zip.stat().st_size / 1e6:.1f} MB")
print("Downloading...")
files.download(str(results_zip))
print("\n✅ monosplat_results.zip downloaded to your computer")


---
## Cell 14 — Debug Commands

Inspect runtime state. Run this cell if something goes wrong.


In [ ]:
import subprocess
import sys
from pathlib import Path

print("=== Git commit ===")
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=False)

print("\n=== Repository status ===")
subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], check=False)

print("\n=== Training script help ===")
subprocess.run([sys.executable, str(TRAIN_ENTRYPOINT), "--help"], check=False)

print("\n=== Recent log tail ===")
if 'TRAIN_LOG' in dir() and Path(str(TRAIN_LOG)).exists():
    print("\n".join(Path(str(TRAIN_LOG)).read_text(encoding='utf-8', errors='replace').splitlines()[-80:]))
else:
    print("No TRAIN_LOG found — training has not run yet in this session.")
